# Practica NoSQL: Integracion de Python y Procesamiento Paralelo

**Bootcamp:** Python PY4  
**Modulo 4:** Big Data y MLOps


>Nota:   
>
>El aula indica que esta practica se puede desarrollar en equipos de 3 integrantes,
>pero tambien es totalmente valido resolverla de forma individual. Elige la modalidad
>con la que te sientas mas comodo.
>
>**Sobre la entrega:** todos deben subir la tarea a Moodle. Si trabajaron en equipo, cada integrante sube el mismo
>archivo.

## Objetivo

Vas a construir, de principio a fin, el mismo tipo de flujo que trabajamos en las sesiones de la semana:

1. Procesar un dataset grande con **Dask** y comparar su rendimiento contra
   **Pandas**.
2. Guardar los resultados procesados en **MongoDB** y ejecutar consultas complejas.
3. (Reto opcional) Respaldar los resultados en almacenamiento cloud.

El dataset de esta practica es distinto al que usamos en clase: en vez de
transacciones bancarias, hoy vamos a trabajar con **pedidos de una tienda en
linea**. La logica es exactamente la misma, solo cambia el contexto de negocio.

Completa las celdas de codigo marcadas con `# TODO`. Las celdas que ya traen codigo
(generacion del dataset, conexiones) no las necesitas modificar, son la base sobre la
que vas a trabajar.


## 0. Preparacion del entorno

Ejecuta esta celda para importar todo lo que vas a necesitar durante la practica.


In [1]:
# Descomenta si necesitas instalar alguna libreria
!pip install dask[dataframe] pyarrow pymongo boto3 moto

import os
import glob
import time

import numpy as np
import pandas as pd
import dask.dataframe as dd

np.random.seed(7)


## 1. Dataset: pedidos de una tienda en linea

Vamos a generar un dataset sintetico de pedidos, particionado en varios archivos CSV,
tal como lo recibirias de un sistema real de e-commerce.

Columnas del dataset:

- `id_pedido`: identificador unico del pedido.
- `fecha`: fecha del pedido durante 2024.
- `monto`: monto total del pedido en dolares.
- `categoria`: categoria del producto principal del pedido.
- `canal`: canal de compra (web, app movil o marketplace).
- `metodo_pago`: metodo de pago usado.
- `ciudad_envio`: ciudad de destino del envio.

### Elige el tamanio del dataset

En la siguiente celda vas a encontrar tres configuraciones de tamanio. **Descomenta
solo una** (la que quieras usar) y deja las otras tres comentadas.

Se puede tratar con tamanio mas grande para ver como Pandas empieza
a sufrir mientras Dask se mantiene estable. Ahi es donde la comparativa de tiempos se
vuelve realmente interesante.

| Configuracion | Filas totales | Espacio en disco | RAM aprox. que necesita Pandas |
|---|---|---|---|
| MEDIANO | 3,000,000 | ~185 MB | ~0.6 GB |
| GRANDE | 8,000,000 | ~500 MB | ~1.6 GB |
| EXTREMO | 24,000,000 | ~1.5 GB | ~4.8 GB |

Ten en cuenta que Pandas necesita bastante mas RAM que el tamanio del archivo en
disco, porque al concatenar los DataFrames mantiene temporalmente dos copias de los
datos en memoria. Si eliges EXTREMO en una maquina con poca RAM, es posible  que
la celda de Pandas falle con un `MemoryError`: eso es
exactamente el problema que Dask viene a resolver.

La generacion del dataset EXTREMO puede tardar varios minutos.


In [2]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

categorias = ["Electronica", "Ropa", "Hogar", "Deportes", "Belleza", "Alimentos"]
canales = ["web", "app_movil", "marketplace"]
metodos_pago = ["tarjeta", "efectivo", "transferencia", "billetera_digital"]
ciudades = ["San Salvador", "Santa Ana", "Soyapango", "Mejicanos", "San Miguel", "Santa Tecla"]

# ---------------------------------------------------------------------------
# TAMANIO DEL DATASET: descomenta SOLO UNA de las siguientes configuraciones
# ---------------------------------------------------------------------------

# MEDIANO: ~3 millones de filas (~185 MB).
N_ARCHIVOS, FILAS_POR_ARCHIVO = 6, 500_000

# GRANDE: ~8 millones de filas (~500 MB). Pandas se vuelve lento y pesado en RAM.
# N_ARCHIVOS, FILAS_POR_ARCHIVO = 8, 1_000_000

# EXTREMO: ~24 millones de filas (~1.5 GB). Pandas puede fallar por falta de memoria.
# N_ARCHIVOS, FILAS_POR_ARCHIVO = 12, 2_000_000

# ---------------------------------------------------------------------------

print(f"Configuracion elegida: {N_ARCHIVOS} archivos x {FILAS_POR_ARCHIVO:,} filas")
print(f"Total de filas a generar: {N_ARCHIVOS * FILAS_POR_ARCHIVO:,}")


def generar_archivo_pedidos(indice):
    n = FILAS_POR_ARCHIVO
    inicio_id = indice * n
    df = pd.DataFrame({
        "id_pedido": np.arange(inicio_id, inicio_id + n),
        "fecha": pd.to_datetime("2024-01-01") + pd.to_timedelta(
            np.random.randint(0, 365, n), unit="D"
        ),
        "monto": np.round(np.random.exponential(scale=45, size=n) + 5, 2),
        "categoria": np.random.choice(categorias, n),
        "canal": np.random.choice(canales, n, p=[0.5, 0.35, 0.15]),
        "metodo_pago": np.random.choice(metodos_pago, n),
        "ciudad_envio": np.random.choice(ciudades, n),
    })
    ruta = os.path.join(DATA_DIR, f"pedidos_2024_part{indice + 1}.csv")
    df.to_csv(ruta, index=False)
    return ruta


# Borramos archivos de una ejecucion anterior, por si cambiaste de tamanio
for archivo_viejo in glob.glob(os.path.join(DATA_DIR, "pedidos_2024_part*.csv")):
    os.remove(archivo_viejo)

t0 = time.time()
rutas = [generar_archivo_pedidos(i) for i in range(N_ARCHIVOS)]
print(f"\nArchivos generados en {time.time() - t0:.1f} segundos:")
espacio_total = 0
for r in rutas:
    tamanio = os.path.getsize(r)
    espacio_total += tamanio
    print(f" - {r} ({tamanio / 1e6:.1f} MB)")
print(f"Espacio total en disco: {espacio_total / 1e6:.1f} MB")


Configuracion elegida: 6 archivos x 500,000 filas
Total de filas a generar: 3,000,000

Archivos generados en 27.6 segundos:
 - data/pedidos_2024_part1.csv (31.3 MB)
 - data/pedidos_2024_part2.csv (31.4 MB)
 - data/pedidos_2024_part3.csv (31.9 MB)
 - data/pedidos_2024_part4.csv (31.9 MB)
 - data/pedidos_2024_part5.csv (31.9 MB)
 - data/pedidos_2024_part6.csv (31.9 MB)
Espacio total en disco: 190.3 MB


## 2. Fase de procesamiento: Pandas vs Dask

El objetivo del analisis es identificar los **pedidos grandes**: aquellos con un
monto mayor a 80 dolares. Vamos a calcular el monto promedio de esos pedidos grandes
por categoria, primero con Pandas y despues con Dask, midiendo el tiempo de cada
enfoque.

### 2.1 Enfoque secuencial con Pandas

Completa el codigo para:
1. Leer todos los archivos CSV de `data/` y concatenarlos en un solo DataFrame.
2. Filtrar los pedidos con `monto > 80`.
3. Agrupar por `categoria` y calcular el promedio de `monto`.
4. Medir cuanto tiempo tarda todo el proceso.


In [3]:
t0 = time.time()

archivos = glob.glob(os.path.join(DATA_DIR, "pedidos_2024_part*.csv"))
df_pandas = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)

df_pandas_grandes = df_pandas[df_pandas["monto"] > 80]

promedio_pandas = (
    df_pandas_grandes.groupby("categoria")["monto"]
    .mean()
    .sort_values(ascending=False)
)

tiempo_pandas = time.time() - t0
print(f"Tiempo Pandas: {tiempo_pandas:.2f} segundos")
print(promedio_pandas)

Tiempo Pandas: 7.14 segundos
categoria
Alimentos      125.277295
Ropa           125.196609
Hogar          125.158907
Belleza        125.095667
Deportes       125.053666
Electronica    124.787014
Name: monto, dtype: float64


### 2.2 Enfoque paralelo con Dask

Ahora repite el mismo analisis, pero usando `dask.dataframe`. Recuerda que las
operaciones de Dask son *lazy*: no se ejecutan hasta que llamas a `.compute()`.


In [4]:
t0 = time.time()

ddf = dd.read_csv(os.path.join(DATA_DIR, "pedidos_2024_part*.csv"))

ddf_grandes = ddf[ddf["monto"] > 80]
promedio_dask_lazy = ddf_grandes.groupby("categoria")["monto"].mean()

promedio_dask = promedio_dask_lazy.compute().sort_values(ascending=False)

tiempo_dask = time.time() - t0
print(f"Tiempo Dask: {tiempo_dask:.2f} segundos")
print(promedio_dask)

Tiempo Dask: 7.62 segundos
categoria
Alimentos      125.277295
Ropa           125.196609
Hogar          125.158907
Belleza        125.095667
Deportes       125.053666
Electronica    124.787014
Name: monto, dtype: float64


### 2.3 Comparativa de tiempos

Completa la tabla comparativa. Este es uno de los puntos clave de evaluacion de la
practica: documentar la diferencia de tiempo entre ambos enfoques.


In [5]:
comparativa = pd.DataFrame({
    "enfoque": ["Pandas (secuencial)", "Dask (paralelo)"],
    "tiempo_segundos": [round(tiempo_pandas, 2), round(tiempo_dask, 2)],
})
comparativa["mas_rapido"] = comparativa["tiempo_segundos"] == comparativa["tiempo_segundos"].min()
comparativa

,enfoque,tiempo_segundos,mas_rapido
0,Pandas (secuencial),7.14,True
1,Dask (paralelo),7.62,False


## 3. Filtro adicional: pedidos por canal digital

Como practica adicional de filtros, identifica los pedidos grandes (monto > 80) que
se hicieron por canales digitales (`app_movil` o `web`), usando Dask.


In [6]:
pedidos_canal_digital = ddf[
    (ddf["monto"] > 80) & (ddf["canal"].isin(["app_movil", "web"]))
].compute()

print(f"Pedidos grandes por canal digital: {len(pedidos_canal_digital):,}")
pedidos_canal_digital.head()

Pedidos grandes por canal digital: 481,316


,id_pedido,fecha,monto,categoria,canal,metodo_pago,ciudad_envio
8,8,2024-01-24,154.74,Deportes,web,tarjeta,Santa Ana
20,20,2024-10-02,83.97,Ropa,web,billetera_digital,Santa Tecla
29,29,2024-07-08,89.54,Hogar,web,efectivo,Santa Tecla
36,36,2024-12-09,83.85,Deportes,app_movil,transferencia,San Salvador
40,40,2024-09-14,100.78,Electronica,web,tarjeta,Santa Tecla


## 4. Preparar los resultados para MongoDB

Antes de conectarnos a MongoDB, guardamos dos resultados que vamos a insertar como
documentos:

1. El reporte agregado de promedio de monto por categoria.
2. Una muestra de 500 pedidos grandes individuales.


In [7]:
reporte_categorias = promedio_dask.reset_index()
reporte_categorias.columns = ["categoria", "promedio_monto"]

muestra_pedidos_grandes = df_pandas_grandes.sample(n=500, random_state=7).copy()
# Convertimos fecha a texto para que sea serializable como documento en MongoDB
muestra_pedidos_grandes["fecha"] = muestra_pedidos_grandes["fecha"].astype(str)

reporte_categorias_dict = reporte_categorias.to_dict("records")
pedidos_grandes_dict = muestra_pedidos_grandes.to_dict("records")

print(f"Reporte por categoria: {len(reporte_categorias_dict)} documentos")
print(f"Muestra de pedidos grandes: {len(pedidos_grandes_dict)} documentos")

Reporte por categoria: 6 documentos
Muestra de pedidos grandes: 500 documentos


## 5. Conexion a MongoDB

1. Crea una cuenta gratuita en `https://www.mongodb.com/cloud/atlas/register`.
2. Crea un cluster gratuito.
3. En "Database Access", crea un usuario con contrasenia.
4. En "Network Access", agrega tu direccion IP (o `0.0.0.0/0` para permitir cualquier
   IP, eso es ok para la clase, nunca para produccion).
5. En "Connect" -> "Drivers", copia el `connection string`. Se ve algo asi:

   ```
   mongodb+srv://<db_username>:<db_password>@cluster0.xxxxx.mongodb.net/?retryWrites=true&w=majority
   ```

6. Reemplaza `<db_username>` y `<db_password>` por el usuario y la contrasenia que
   creaste en el paso 3. Ojo: hay que borrar tambien los simbolos `<` y `>`.



In [8]:
!pip install --upgrade pymongo certifi -q

import certifi
from pymongo import MongoClient, DESCENDING

MONGO_URI = "<TU_MONGODB_URI>"

# tlsCAFile=certifi.where() evita el error de "SSL handshake failed" comun en Colab
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())

try:
    client.admin.command("ping")
    print("Conexion exitosa a MongoDB")
except Exception as error:
    print("No se pudo conectar a MongoDB. Revisa tu MONGO_URI.")
    print("Detalle del error:", error)

Conexion exitosa a MongoDB


In [11]:
db = client["tienda_db"]
coleccion_reporte = db["reporte_categorias"]
coleccion_pedidos = db["pedidos_grandes"]

# Limpiamos por si se vuelve a correr el notebook (evita duplicados)
coleccion_reporte.delete_many({})
coleccion_pedidos.delete_many({})

if reporte_categorias_dict:
    coleccion_reporte.insert_many(reporte_categorias_dict)
if pedidos_grandes_dict:
    coleccion_pedidos.insert_many(pedidos_grandes_dict)

print(f"Documentos en reporte_categorias: {coleccion_reporte.count_documents({})}")
print(f"Documentos en pedidos_grandes: {coleccion_pedidos.count_documents({})}")

Documentos en reporte_categorias: 6
Documentos en pedidos_grandes: 500


## 6. Consultas complejas en MongoDB

Completa las siguientes tres consultas sobre la coleccion `pedidos_grandes`. Esta
seccion es el otro punto clave de evaluacion de la practica.

### 6.1 Consulta 1: filtro con multiples condiciones

Pedidos con `monto` mayor a 150 realizados por el canal `app_movil`.


In [12]:
consulta1 = list(
    coleccion_pedidos.find({"monto": {"$gt": 150}, "canal": "app_movil"})
)
print(f"Pedidos con monto > 150 en canal app_movil: {len(consulta1)}")
for doc in consulta1[:5]:
    print(doc)

Pedidos con monto > 150 en canal app_movil: 35
{'_id': ObjectId('6ab088b3b866b22f59bfa21e'), 'id_pedido': 1058246, 'fecha': '2024-03-21', 'monto': 224.76, 'categoria': 'Ropa', 'canal': 'app_movil', 'metodo_pago': 'efectivo', 'ciudad_envio': 'San Miguel'}
{'_id': ObjectId('6ab088b3b866b22f59bfa229'), 'id_pedido': 2023197, 'fecha': '2024-03-29', 'monto': 153.46, 'categoria': 'Hogar', 'canal': 'app_movil', 'metodo_pago': 'transferencia', 'ciudad_envio': 'Santa Ana'}
{'_id': ObjectId('6ab088b3b866b22f59bfa235'), 'id_pedido': 1237018, 'fecha': '2024-06-06', 'monto': 161.0, 'categoria': 'Deportes', 'canal': 'app_movil', 'metodo_pago': 'tarjeta', 'ciudad_envio': 'Santa Ana'}
{'_id': ObjectId('6ab088b3b866b22f59bfa241'), 'id_pedido': 902996, 'fecha': '2024-03-14', 'monto': 165.52, 'categoria': 'Ropa', 'canal': 'app_movil', 'metodo_pago': 'billetera_digital', 'ciudad_envio': 'Mejicanos'}
{'_id': ObjectId('6ab088b3b866b22f59bfa254'), 'id_pedido': 536620, 'fecha': '2024-06-02', 'monto': 158.57, '

### 6.2 Consulta 2: ordenamiento y limite

Los 5 pedidos de mayor monto de toda la coleccion.


In [13]:
consulta2 = list(
    coleccion_pedidos.find().sort("monto", DESCENDING).limit(5)
)
for doc in consulta2:
    print(doc)

{'_id': ObjectId('6ab088b3b866b22f59bfa3ac'), 'id_pedido': 235279, 'fecha': '2024-07-17', 'monto': 511.25, 'categoria': 'Ropa', 'canal': 'app_movil', 'metodo_pago': 'transferencia', 'ciudad_envio': 'Santa Tecla'}
{'_id': ObjectId('6ab088b3b866b22f59bfa386'), 'id_pedido': 459121, 'fecha': '2024-10-07', 'monto': 314.52, 'categoria': 'Electronica', 'canal': 'marketplace', 'metodo_pago': 'transferencia', 'ciudad_envio': 'Santa Ana'}
{'_id': ObjectId('6ab088b3b866b22f59bfa326'), 'id_pedido': 1182446, 'fecha': '2024-11-23', 'monto': 312.73, 'categoria': 'Electronica', 'canal': 'app_movil', 'metodo_pago': 'billetera_digital', 'ciudad_envio': 'Soyapango'}
{'_id': ObjectId('6ab088b3b866b22f59bfa293'), 'id_pedido': 1008756, 'fecha': '2024-08-20', 'monto': 301.06, 'categoria': 'Ropa', 'canal': 'web', 'metodo_pago': 'efectivo', 'ciudad_envio': 'Santa Ana'}
{'_id': ObjectId('6ab088b3b866b22f59bfa306'), 'id_pedido': 1296506, 'fecha': '2024-05-16', 'monto': 287.92, 'categoria': 'Alimentos', 'canal': 

### 6.3 Consulta 3: agregacion con `$group`

Promedio de monto y cantidad de pedidos por metodo de pago, ordenado de mayor a
menor promedio.


In [14]:
pipeline = [
    {"$group": {
        "_id": "$metodo_pago",
        "promedio_monto": {"$avg": "$monto"},
        "cantidad_pedidos": {"$sum": 1},
    }},
    {"$sort": {"promedio_monto": -1}},
]
consulta3 = list(coleccion_pedidos.aggregate(pipeline))
for doc in consulta3:
    print(doc)

{'_id': 'billetera_digital', 'promedio_monto': 125.79025862068966, 'cantidad_pedidos': 116}
{'_id': 'transferencia', 'promedio_monto': 124.90482014388489, 'cantidad_pedidos': 139}
{'_id': 'tarjeta', 'promedio_monto': 124.5612213740458, 'cantidad_pedidos': 131}
{'_id': 'efectivo', 'promedio_monto': 124.25868421052631, 'cantidad_pedidos': 114}


In [15]:
# Cerramos la conexion al terminar
client.close()
print("Conexion cerrada")


Conexion cerrada


## 7. Reto opcional: respaldo en la nube

Si quieres practicar tambien la parte de Cloud Storage, sube el reporte de
categorias a un bucket simulado con `moto`, igual que hicimos en la Sesion 3. Esta
seccion es opcional y no es obligatoria para completar la practica.


In [16]:
# TODO (opcional): sube reporte_categorias a un bucket simulado con moto
import boto3
from moto import mock_aws

